## 面试问题

ReAct 式 Thought–Action–Observation 交织与直接函数调用循环怎么选？

## 回答主线

ReAct 每步显式产出自然语言 Thought 再给 Action，函数调用循环则直接吐结构化 tool call。本 Notebook 在同一个「两跳问答」任务上并排实现两种循环，验证三点：(1) 两者都能靠回读观察逐跳逼近答案；(2) ReAct 因为多了 Thought 文本而更费 token；(3) Thought 只是文本、不是承诺——当 Thought 与 Action 不一致时，必须以真正执行的 Action 和 Observation 为准。

## 真实案例

多跳问答：先查产品所属团队，再查该团队负责人。知识用一张 4 条记录的离线表 `(kind, key) -> value`。ReAct 轨迹每步带 Thought，函数调用轨迹只有 tool call。数据为离线小表，只解释循环结构，不代表真实模型推理能力。

In [1]:
knowledge = {  # 定义离线多跳知识表。
    ("product", "wombat"): "team-alpha",  # 产品 wombat 属于 team-alpha。
    ("team", "team-alpha"): "dr-lee",  # team-alpha 的负责人是 dr-lee。
    ("product", "gecko"): "team-beta",  # 产品 gecko 属于 team-beta。
    ("team", "team-beta"): "dr-fan",  # team-beta 的负责人是 dr-fan。
}  # 结束知识表定义。

question = "产品 wombat 的团队负责人是谁"  # 定义需要两跳才能回答的问题。

def tool_lookup(kind, key):  # 定义唯一的查表工具。
    return knowledge.get((kind, key), "unknown")  # 返回命中值或 unknown。

print("知识表条目数:", len(knowledge))  # 展示知识表规模。
print("问题:", question)  # 展示多跳问题。

知识表条目数: 4
问题: 产品 wombat 的团队负责人是谁


## 基线（Baseline）

先做函数调用循环作为基线：模型每步只输出一个结构化 tool call，没有自然语言推理，靠回读的观察决定下一跳。

In [2]:
def fc_policy(state):  # 函数调用策略：根据状态决定下一个工具调用。
    if state["team"] is None:  # 还不知道团队就先查产品归属。
        return ("lookup", "product", state["product"])  # 提议查询产品归属。
    if state["lead"] is None:  # 知道团队但不知负责人就查负责人。
        return ("lookup", "team", state["team"])  # 提议查询团队负责人。
    return ("finish", None, None)  # 两跳都完成则结束。

def run_fc(product, max_steps=5):  # 运行函数调用循环。
    state = {"product": product, "team": None, "lead": None}  # 初始化多跳状态。
    trace = []  # 记录工具调用轨迹。
    for _ in range(max_steps):  # 用步数上限约束循环。
        call = fc_policy(state)  # 让策略提议一个动作。
        if call[0] == "finish":  # 命中结束动作就停止。
            break  # 退出循环。
        result = tool_lookup(call[1], call[2])  # 执行查表工具。
        trace.append((call, result))  # 记录本步调用与结果。
        if call[1] == "product":  # 查产品归属得到团队。
            state["team"] = result  # 回写团队。
        else:  # 否则查到的是负责人。
            state["lead"] = result  # 回写负责人。
    return state, trace  # 返回终态与轨迹。

fc_state, fc_trace = run_fc("wombat")  # 在 wombat 上运行函数调用循环。
print("函数调用答案:", fc_state["lead"], "| 工具步数:", len(fc_trace))  # 展示最终负责人与步数。
for call, result in fc_trace:  # 逐条打印工具调用轨迹。
    print("call", call, "->", result)  # 展示每步调用与结果。

函数调用答案: dr-lee | 工具步数: 2
call ('lookup', 'product', 'wombat') -> team-alpha
call ('lookup', 'team', 'team-alpha') -> dr-lee


## 核心实现：ReAct 交织循环

ReAct 在每步动作前多产出一个自然语言 Thought，把「为什么这么做」显式化，便于审计和多跳纠错。动作与观察机制和函数调用循环相同。

In [3]:
def react_policy(state):  # ReAct 策略：先产出思考再给动作。
    if state["team"] is None:  # 尚不知道团队时的推理分支。
        thought = "还不知道产品所属团队，先查产品归属"  # 生成自然语言思考。
        return thought, ("lookup", "product", state["product"])  # 返回思考与动作。
    if state["lead"] is None:  # 已知团队但未知负责人的推理分支。
        thought = "已知团队，接下来查该团队负责人"  # 生成自然语言思考。
        return thought, ("lookup", "team", state["team"])  # 返回思考与动作。
    thought = "两跳信息齐备，可以作答"  # 完成时的思考。
    return thought, ("finish", None, None)  # 返回思考与结束动作。

def run_react(product, max_steps=5):  # 运行 ReAct 循环。
    state = {"product": product, "team": None, "lead": None}  # 初始化多跳状态。
    trace = []  # 记录带思考的轨迹。
    for _ in range(max_steps):  # 步数上限约束循环。
        thought, call = react_policy(state)  # 策略产出思考与动作。
        if call[0] == "finish":  # 结束动作则停止。
            trace.append((thought, call, None))  # 记录最后一步思考。
            break  # 退出循环。
        result = tool_lookup(call[1], call[2])  # 执行工具。
        trace.append((thought, call, result))  # 记录思考、动作与观察。
        if call[1] == "product":  # 产品归属回写团队。
            state["team"] = result  # 回写团队。
        else:  # 否则回写负责人。
            state["lead"] = result  # 回写负责人。
    return state, trace  # 返回终态与轨迹。

react_state, react_trace = run_react("wombat")  # 在 wombat 上运行 ReAct 循环。
executed_steps = len([t for t in react_trace if t[2] is not None])  # 统计真正执行动作的步数。
print("ReAct 答案:", react_state["lead"], "| 执行步数:", executed_steps)  # 展示答案与执行步数。
for thought, call, result in react_trace:  # 逐条打印带思考的轨迹。
    print("思考:", thought, "| 动作:", call, "->", result)  # 展示每步思考、动作与观察。

ReAct 答案: dr-lee | 执行步数: 2
思考: 还不知道产品所属团队，先查产品归属 | 动作: ('lookup', 'product', 'wombat') -> team-alpha
思考: 已知团队，接下来查该团队负责人 | 动作: ('lookup', 'team', 'team-alpha') -> dr-lee
思考: 两跳信息齐备，可以作答 | 动作: ('finish', None, None) -> None


In [4]:
def rough_tokens(trace, has_thought):  # 粗略估算轨迹的 token 规模。
    tokens = 0  # 初始化计数。
    for row in trace:  # 遍历轨迹每一步。
        tokens += 6  # 每个动作与观察约计 6 个 token。
        if has_thought:  # ReAct 额外包含自然语言思考。
            tokens += len(str(row[0]))  # 用思考文本长度粗估其 token。
    return tokens  # 返回粗估 token 数。

fc_tokens = rough_tokens(fc_trace, has_thought=False)  # 估算函数调用轨迹 token。
react_tokens = rough_tokens(react_trace, has_thought=True)  # 估算 ReAct 轨迹 token。
print("token 粗估 -> 函数调用:", fc_tokens, "| ReAct:", react_tokens)  # 对比两者开销。

token 粗估 -> 函数调用: 12 | ReAct: 61


## 结果解读

两种循环都得到 `dr-lee`：函数调用用 2 步工具调用，ReAct 也用 2 步执行但多了 3 段 Thought。token 粗估显示 ReAct 明显更贵——多出的自然语言是可解释性的成本。是否值得，取决于任务是否需要审计推理链或多跳纠错。

## 失败案例与修正

关键风险：Thought 与 Action 不一致。下面构造一个坏策略，Thought 声称「要查团队负责人」，Action 却仍在查产品归属。修正原则很简单但常被违反——**以真正执行的 Action 和它的 Observation 为准，绝不采信 Thought 的自我声称**。

In [5]:
def inconsistent_policy(state):  # 构造思考与动作不一致的坏策略。
    thought = "我应该去查团队负责人"  # 思考声称要查负责人。
    return thought, ("lookup", "product", state["product"])  # 动作却仍在查产品归属。

bad_state = {"product": "wombat", "team": None, "lead": None}  # 构造初始状态。
bad_thought, bad_call = inconsistent_policy(bad_state)  # 取出不一致的思考与动作。
executed = tool_lookup(bad_call[1], bad_call[2])  # 以真正执行的动作为准查表。
print("思考声称:", bad_thought)  # 展示模型的自然语言声称。
print("实际执行动作:", bad_call, "-> 观察:", executed)  # 展示真正发生的动作与观察。

思考声称: 我应该去查团队负责人
实际执行动作: ('lookup', 'product', 'wombat') -> 观察: team-alpha


In [6]:
assert fc_state["lead"] == "dr-lee"  # 函数调用循环应得到正确负责人。
assert react_state["lead"] == "dr-lee"  # ReAct 循环应得到相同答案。
assert len(fc_trace) == 2  # 两跳任务函数调用应恰好两步工具调用。
assert react_tokens > fc_tokens  # ReAct 因含思考文本应更费 token。
assert executed == "team-alpha"  # 以动作为准得到的是产品归属而非负责人。
assert bad_call[1] == "product"  # 证明实际动作与思考声称不一致。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
